## Structured Error Analysis

The goal of structured error analysis is not only to measure performance numerically, but also to understand the types of mistakes made by each model. 

In particular, this analysis focuses on:

- which classes are most commonly confused
- whether the models fail on the same examples
- what linguistic or contextual properties make certain tweets difficult to classify

This will help provide us with a deeper insight into model behavior beyond overall accuracy and macro-averaged metrics.

### Load saved Artifacts

In [2]:
import joblib

# Load train-test split
x_train = joblib.load(r".\Artifacts\x_train.pkl")
x_test = joblib.load(r".\Artifacts\x_test.pkl")
y_train = joblib.load(r".\Artifacts\y_train.pkl")
y_test = joblib.load(r".\Artifacts\y_test.pkl")
class_names = joblib.load(r".\Artifacts\class_names.pkl")

print("Data split loaded")

# Load Naive Bayes
nb_model = joblib.load(r".\Artifacts\naive_bayes_model.pkl")
nb_vectorizer = joblib.load(r".\Artifacts\naive_bayes_vectorizer.pkl")

# Load Logistic Regression
lr_model = joblib.load(r".\Artifacts\logistic_regression_model.pkl")
lr_vectorizer = joblib.load(r".\Artifacts\logistic_regression_vectorizer.pkl")

# Load SVM
svm_model = joblib.load(r".\Artifacts\svm_model.pkl")
svm_vectorizer = joblib.load(r".\Artifacts\svm_vectorizer.pkl")

print("NB, LR, SVM model loaded")
print("NB, LR, SVM vectorizer loaded")

Data split loaded
NB, LR, SVM model loaded
NB, LR, SVM vectorizer loaded


### Recreate Predictions on the Test Set

In [7]:
models = {
    "NB": (nb_model, nb_vectorizer),
    "LR": (lr_model, lr_vectorizer),
    "SVM": (svm_model, svm_vectorizer)
}

predictions = {}

for name, (model, vectorizer) in models.items():
    x_test_vecs = vectorizer.transform(x_test)
    predictions[name] = model.predict(x_test_vecs)
    print(f"{name} predictions generated")

NB predictions generated
LR predictions generated
SVM predictions generated


## Master Error Analysis Table

This table consolidates the ground truth labels, model predictions, and correctness indicators across all models at the instance level.

It serves as the foundation for structured error analysis by enabling:

- Identification of systematic misclassification patterns (e.g., specific label confusions)
- Direct comparison of model behaviors on the same input instances
- Isolation of cases where models agree or disagree in their errors
- Extraction of representative error samples for qualitative analysis

This unified view allows us to move beyond aggregate metrics (e.g., accuracy, precision) and instead analyze **how** and **why** models fail.

In [8]:
import pandas as pd

error_df = pd.DataFrame({
    "tweet": x_test,
    "true_label": [class_names[i] for i in y_test]
})

for model_name, y_pred in predictions.items():
    preds = []
    for i in y_pred:
        preds.append(class_names[i])
        
    error_df[f"{model_name}_pred"] = preds
    error_df[f"{model_name}_correct"] = error_df["true_label"] == error_df[f"{model_name}_pred"]

error_df.head()

,tweet,true_label,NB_pred,NB_correct,LR_pred,LR_correct,SVM_pred,SVM_correct
0,today tomorrow try move mountains cross fingers,other_cyberbullying,not_cyberbullying,False,not_cyberbullying,False,other_cyberbullying,True
1,typa girl bullied lesbians high school,age,age,True,age,True,age,True
2,cool woman gave directions polling place whose...,age,age,True,age,True,age,True
3,need teach version quran terrorists killing pe...,religion,religion,True,religion,True,religion,True
4,rome destroy coliseum roman forum every buildi...,religion,religion,True,religion,True,religion,True


### Error Partitioning and Dataset Difficulty Analysis

To better understand model behavior beyond aggregate performance metrics, we partition the dataset based on how many models correctly classify each sample.

For each tweet, we compute the number of models that predict the correct label. Using this, we divide the dataset into three categories:

- **All Correct**: Tweets correctly classified by all models  
- **All Wrong**: Tweets misclassified by all models  
- **Mixed**: Tweets where some models are correct and others are incorrect  

This partitioning allows us to approximate the difficulty of the dataset:

- Tweets in the *All Correct* category are likely easier and contain clear patterns that all models can learn.  
- Tweets in the *All Wrong* category represent difficult or ambiguous cases that current models consistently fail to capture.  
- The *Mixed* category is particularly important, as it highlights disagreement between models and reveals differences in their learning behavior.

By structuring the data in this way, we move from evaluating overall accuracy to analyzing where and why models succeed or fail, which forms the basis for deeper error analysis in subsequent sections.

In [14]:
model_names = list(predictions.keys())

# Count how many models got it correct per row
error_df["num_correct"] = error_df[[f"{m}_correct" for m in model_names]].sum(axis=1)

# Count how many got it wrong
error_df["num_wrong"] = len(model_names) - error_df["num_correct"]

# All models correct
all_correct = error_df[error_df["num_correct"] == len(model_names)]

# All models wrong
all_wrong = error_df[error_df["num_correct"] == 0]

# Mixed (some correct, some wrong)
mixed = error_df[
    (error_df["num_correct"] > 0) &
    (error_df["num_correct"] < len(model_names))
]

model_strengths = {}

for m in model_names:
    model_strengths[m] = error_df[
        (error_df[f"{m}_correct"] == True) &
        (error_df["num_correct"] < len(model_names))  # others failed
    ]

model_weaknesses = {}

for m in model_names:
    model_weaknesses[m] = error_df[
        (error_df[f"{m}_correct"] == False) &
        (error_df["num_correct"] > 0)  # others succeeded
    ]

summary = pd.DataFrame({
    "Total Samples": [len(error_df)],
    "All Correct": [len(all_correct)],
    "All Wrong": [len(all_wrong)],
    "Mixed": [len(mixed)]
})

summary

,Total Samples,All Correct,All Wrong,Mixed
0,11923,8174,1622,2127


#### Observations from Error Partitioning

- A majority of samples (8174 / 11923) are correctly classified by all models, suggesting that a large portion of the dataset contains clear and easily distinguishable patterns.

- However, 1622 samples are misclassified by all models, indicating the presence of inherently difficult or ambiguous cases that current models struggle to capture.

- Notably, 2127 samples fall into the mixed category, where models disagree in their predictions. These cases are particularly important as they reveal differences in model behavior and will be further analyzed to understand model-specific strengths and weaknesses.

- The existence of a substantial mixed category suggests that different models capture different aspects of the data, rather than one model universally outperforming others.